# MMSegmentation Tutorial

This version runs with the **pure-PyTorch port** — no mmengine/mmcv packages required.
Simply run cells top-to-bottom inside the repo directory.

Topics covered:
* Inference with a pretrained checkpoint
* Fine-tuning on the Stanford Background dataset
* Visualising results

## 1  Environment setup

Add the repo root to `sys.path` so the in-repo `mmengine/` and `mmcv/` shims shadow any installed packages.

In [ ]:
import os, sys

# demo/ is one level below the repo root
_REPO_ROOT = os.path.dirname(os.path.abspath('.'))
if _REPO_ROOT not in sys.path:
    sys.path.insert(0, _REPO_ROOT)

import torch
import mmengine, mmcv, mmseg

print('PyTorch  :', torch.__version__, '  CUDA available:', torch.cuda.is_available())
print('mmengine :', mmengine.__version__)
print('mmcv     :', mmcv.__version__)
print('mmseg    :', mmseg.__version__)
print('Repo root:', _REPO_ROOT)

## 2  Inference with a pretrained model

We use **PSPNet-R50-D8** trained on Cityscapes (77.85 mIoU).

In [ ]:
import urllib.request

ckpt_dir  = os.path.join(_REPO_ROOT, 'checkpoints')
ckpt_name = 'pspnet_r50-d8_512x1024_40k_cityscapes_20200605_003338-2966598c.pth'
ckpt_path = os.path.join(ckpt_dir, ckpt_name)
ckpt_url  = ('https://download.openmmlab.com/mmsegmentation/v0.5/pspnet/'
             'pspnet_r50-d8_512x1024_40k_cityscapes/' + ckpt_name)

os.makedirs(ckpt_dir, exist_ok=True)
if not os.path.isfile(ckpt_path):
    print('Downloading checkpoint …')
    urllib.request.urlretrieve(ckpt_url, ckpt_path)
    print(f'Saved to {ckpt_path}')
else:
    print('Checkpoint already present.')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

from mmengine.model import revert_sync_batchnorm
from mmseg.apis import init_model, inference_model
from mmseg.utils import get_classes, get_palette

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'

config_file = os.path.join(_REPO_ROOT, 'configs/pspnet/pspnet_r50-d8_4xb2-40k_cityscapes-512x1024.py')
img_path    = os.path.join(_REPO_ROOT, 'demo/demo.png')

model = init_model(config_file, ckpt_path, device=device)
if not torch.cuda.is_available():
    model = revert_sync_batchnorm(model)
print('Model:', type(model).__name__)

result  = inference_model(model, img_path)
seg_map = result.pred_sem_seg.data.squeeze(0).cpu().numpy().astype(np.int32)
print('Seg map shape:', seg_map.shape, ' labels:', sorted(np.unique(seg_map).tolist()))

In [ ]:
dataset_meta = getattr(model, 'dataset_meta', {})
classes = dataset_meta.get('classes', get_classes('cityscapes'))
palette = dataset_meta.get('palette', get_palette('cityscapes'))

h, w = seg_map.shape
color_map = np.zeros((h, w, 3), dtype=np.uint8)
for lbl, color in enumerate(palette):
    color_map[seg_map == lbl] = color

orig = np.array(Image.open(img_path).convert('RGB'))
if color_map.shape[:2] != orig.shape[:2]:
    color_map = np.array(Image.fromarray(color_map).resize(
        (orig.shape[1], orig.shape[0]), Image.NEAREST))
alpha   = 0.5
blended = ((1 - alpha) * orig + alpha * color_map).clip(0, 255).astype(np.uint8)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].imshow(orig);       axes[0].set_title('Original');        axes[0].axis('off')
axes[1].imshow(color_map);  axes[1].set_title('Segmentation');     axes[1].axis('off')
axes[2].imshow(blended);    axes[2].set_title('Overlay (α=0.5)'); axes[2].axis('off')
plt.tight_layout(); plt.show()

## 3  Fine-tune on Stanford Background Dataset

The dataset contains 715 outdoor images with 8 classes:  
*sky, tree, road, grass, water, building, mountain, foreground object*

In [ ]:
import subprocess, os

data_root = os.path.join(_REPO_ROOT, 'demo', 'iccv09Data')
tarball   = os.path.join(_REPO_ROOT, 'demo', 'stanford_background.tar.gz')

if not os.path.isdir(data_root):
    print('Downloading Stanford Background Dataset …')
    urllib.request.urlretrieve(
        'http://dags.stanford.edu/data/iccv09Data.tar.gz', tarball)
    subprocess.run(['tar', 'xf', tarball, '-C', os.path.dirname(tarball)], check=True)
    print('Extracted to', data_root)
else:
    print('Dataset already present.')

img_dir = 'images'
ann_dir = 'labels'

In [ ]:
# Quick look at a sample image
sample_img = np.array(Image.open(os.path.join(data_root, 'images/6000124.jpg')).convert('RGB'))
plt.figure(figsize=(8, 5))
plt.imshow(sample_img)
plt.title('Sample image'); plt.axis('off'); plt.show()

In [ ]:
# Class definitions
classes  = ('sky', 'tree', 'road', 'grass', 'water', 'bldg', 'mntn', 'fg obj')
palette  = [[128, 128, 128], [129, 127, 38], [120, 69, 125], [53, 125, 34],
            [0,   11,  123], [118,  20,  12], [122,  81,  25], [241, 134,  51]]

# Convert .regions.txt → .png palette images
import mmengine as _mme
converted = 0
for fname in _mme.scandir(os.path.join(data_root, ann_dir), suffix='.regions.txt'):
    src = os.path.join(data_root, ann_dir, fname)
    dst = src.replace('.regions.txt', '.png')
    if not os.path.isfile(dst):
        seg_map = np.loadtxt(src).astype(np.uint8)
        seg_img = Image.fromarray(seg_map).convert('P')
        seg_img.putpalette(np.array(palette, dtype=np.uint8))
        seg_img.save(dst)
        converted += 1
print(f'Converted {converted} annotation files (0 = already done).')

In [ ]:
import matplotlib.patches as mpatches

ann_img = Image.open(os.path.join(data_root, 'labels/6000124.png'))
fig, ax = plt.subplots(figsize=(8, 5))
ax.imshow(np.array(ann_img.convert('RGB')))
patches = [mpatches.Patch(color=np.array(palette[i])/255., label=classes[i])
           for i in range(8)]
ax.legend(handles=patches, bbox_to_anchor=(1.05, 1), loc=2,
          borderaxespad=0., fontsize='large')
ax.axis('off'); plt.tight_layout(); plt.show()

In [ ]:
import os.path as osp

split_dir = os.path.join(data_root, 'splits')
mmengine.mkdir_or_exist(split_dir)

filename_list = [
    osp.splitext(f)[0]
    for f in mmengine.scandir(os.path.join(data_root, ann_dir), suffix='.png')
]
train_len = int(len(filename_list) * 4 / 5)

with open(osp.join(split_dir, 'train.txt'), 'w') as f:
    f.writelines(line + '\n' for line in filename_list[:train_len])
with open(osp.join(split_dir, 'val.txt'), 'w') as f:
    f.writelines(line + '\n' for line in filename_list[train_len:])

print(f'Train: {train_len}  Val: {len(filename_list) - train_len}')

### 3.1  Register the custom dataset class

In [ ]:
from mmseg.registry import DATASETS
from mmseg.datasets import BaseSegDataset

@DATASETS.register_module(force=True)
class StanfordBackgroundDataset(BaseSegDataset):
    METAINFO = dict(classes=classes, palette=palette)
    def __init__(self, **kwargs):
        super().__init__(img_suffix='.jpg', seg_map_suffix='.png', **kwargs)

print('StanfordBackgroundDataset registered.')

### 3.2  Build and modify the training config

In [ ]:
from mmengine import Config

cfg = Config.fromfile(os.path.join(
    _REPO_ROOT, 'configs/pspnet/pspnet_r50-d8_4xb2-40k_cityscapes-512x1024.py'))

# BN instead of SyncBN (single GPU)
cfg.norm_cfg = dict(type='BN', requires_grad=True)
cfg.crop_size = (256, 256)
cfg.model.data_preprocessor.size = cfg.crop_size
cfg.model.backbone.norm_cfg = cfg.norm_cfg
cfg.model.decode_head.norm_cfg = cfg.norm_cfg
cfg.model.auxiliary_head.norm_cfg = cfg.norm_cfg

# 8 classes instead of 19
cfg.model.decode_head.num_classes = 8
cfg.model.auxiliary_head.num_classes = 8

# Dataset
cfg.dataset_type = 'StanfordBackgroundDataset'
cfg.data_root = data_root

cfg.train_pipeline = [
    dict(type='LoadImageFromFile'),
    dict(type='LoadAnnotations'),
    dict(type='RandomResize', scale=(320, 240), ratio_range=(0.5, 2.0), keep_ratio=True),
    dict(type='RandomCrop', crop_size=cfg.crop_size, cat_max_ratio=0.75),
    dict(type='RandomFlip', prob=0.5),
    dict(type='PackSegInputs'),
]
cfg.test_pipeline = [
    dict(type='LoadImageFromFile'),
    dict(type='Resize', scale=(320, 240), keep_ratio=True),
    dict(type='LoadAnnotations'),
    dict(type='PackSegInputs'),
]

for split, pipeline in [('train', cfg.train_pipeline), ('val', cfg.test_pipeline)]:
    loader = cfg.train_dataloader if split == 'train' else cfg.val_dataloader
    loader.dataset.type = cfg.dataset_type
    loader.dataset.data_root = cfg.data_root
    loader.dataset.data_prefix = dict(img_path=img_dir, seg_map_path=ann_dir)
    loader.dataset.pipeline = pipeline
    loader.dataset.ann_file = f'splits/{split}.txt'

cfg.train_dataloader.batch_size = 8
cfg.test_dataloader = cfg.val_dataloader

# Pretrained weights
cfg.load_from = ckpt_path

# Output
cfg.work_dir = os.path.join(_REPO_ROOT, 'demo/work_dirs/tutorial')

# Short training run
cfg.train_cfg.max_iters = 200
cfg.train_cfg.val_interval = 200
cfg.default_hooks.logger.interval = 10
cfg.default_hooks.checkpoint.interval = 200
cfg['randomness'] = dict(seed=0)

print('Config built successfully.')
print(cfg.pretty_text)

### 3.3  Train

In [ ]:
from mmengine.runner import Runner

runner = Runner.from_cfg(cfg)
runner.train()

### 3.4  Inference with fine-tuned model

In [ ]:
finetuned_ckpt = os.path.join(cfg.work_dir, 'iter_200.pth')

ft_model = init_model(cfg, finetuned_ckpt, device=device)
if not torch.cuda.is_available():
    ft_model = revert_sync_batchnorm(ft_model)

test_img_path = os.path.join(data_root, 'images/6000124.jpg')
ft_result  = inference_model(ft_model, test_img_path)
ft_seg_map = ft_result.pred_sem_seg.data.squeeze(0).cpu().numpy().astype(np.int32)
print('Seg map shape:', ft_seg_map.shape)

In [ ]:
h, w = ft_seg_map.shape
ft_color_map = np.zeros((h, w, 3), dtype=np.uint8)
for lbl, color in enumerate(palette):
    ft_color_map[ft_seg_map == lbl] = color

ft_orig = np.array(Image.open(test_img_path).convert('RGB'))
if ft_color_map.shape[:2] != ft_orig.shape[:2]:
    ft_color_map = np.array(Image.fromarray(ft_color_map).resize(
        (ft_orig.shape[1], ft_orig.shape[0]), Image.NEAREST))
ft_blend = ((1 - alpha) * ft_orig + alpha * ft_color_map).clip(0, 255).astype(np.uint8)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].imshow(ft_orig);       axes[0].set_title('Original');         axes[0].axis('off')
axes[1].imshow(ft_color_map);  axes[1].set_title('Segmentation');      axes[1].axis('off')
axes[2].imshow(ft_blend);      axes[2].set_title('Overlay (α=0.5)');  axes[2].axis('off')

# Legend
handles = [mpatches.Patch(color=np.array(palette[i])/255., label=classes[i])
           for i in range(8)]
axes[1].legend(handles=handles, bbox_to_anchor=(1.05, 1), loc=2,
               borderaxespad=0., fontsize='small')
plt.tight_layout(); plt.show()